# Benchmarking AI weather ensembles on extremes: heatwaves & heavy rain in France

This notebook is a **template benchmark**. It is runnable end-to-end on *synthetic* data (so you can see every plot and check the logic), and every real-world ingredient plugs in through a small registry:

| You want to change... | You touch... |
|---|---|
| a model (AIFS-ENS, Aurora, altas, IFS-ENS...) | `FORECASTS[...]` loader (section 1) |
| the "truth" (ERA5, E-OBS, COMEPHORE, IMERG...) | `OBS[...]` loader (section 1) |
| what counts as extreme | an `EventDef` in `EVENTS` (section 3) |
| a score | add a function to `SAMPLE_METRICS` (section 4) |

**Pipeline**

1. Load forecasts + observations into one *data contract*
2. Equalise ensemble size
3. Build local, seasonal, percentile-based thresholds and binary event indicators
4. Score per (init, lead) *then* reduce, so that block bootstrap and paired tests are cheap
5. Diagnostics: skill vs lead, reliability, performance diagram, FSS, tail QQ, maps, case study

> **The models in this notebook are fake.** `IFS-ENS`, `AIFS-ENS`, `Aurora-ENS`, `altas` are synthetic generators with different skill/spread/tail-damping. Nothing here says anything about the real models until you plug in real data.

## 0. Setup and configuration

In [1]:
import warnings
from dataclasses import dataclass
from typing import Optional

import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter, gaussian_filter
from scipy.signal import lfilter

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(42)

CFG = dict(
    # domain (metropolitan France; apply a land/country mask for real runs)
    lat=(41.0, 51.5), lon=(-5.5, 10.0),
    # climatology used for thresholds. Must NOT overlap your test period if you
    # want a strict out-of-sample setup (ERA5 1991-2020 is the WMO normal).
    ref_period=("1991-01-01", "2020-12-31"),
    season_months=[6, 7, 8],          # evaluate only on valid dates in these months
    q=0.95,                           # percentile defining "extreme"
    window=7,                         # +/- days around each calendar day for the climatology
    wet_thr=1.0,                      # mm/day: wet-day definition
    heat_days=3,                      # min consecutive exceedance days for a heatwave
    n_members=None,                   # None -> use the smallest ensemble among models
    pths=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],   # probability decision thresholds
    pth_main=0.3,                     # the one shown in SEDI-vs-lead
    n_boot=300, block=7,              # block bootstrap over init dates
    max_lead=10,                      # days
    baseline="IFS-ENS",               # reference model for paired differences
)

ModuleNotFoundError: No module named 'matplotlib'

## 1. Data contract and loaders

Everything downstream assumes **daily** fields on a **common grid**:

**Forecast** `xr.Dataset`
* dims: `(init_time, lead_day, member, latitude, longitude)`, latitude ascending
* `lead_day = 0, 1, 2, ...` is the forecast for **calendar date `init_time + lead_day`** (for 00 UTC inits, lead_day 0 = hours 0-24)
* variables: `tmax` (°C, daily max) and `tp` (mm, 24 h accumulation)

**Observations** `xr.Dataset`: dims `(time, latitude, longitude)`, same variables and units, covering the reference period *and* the test period.

Things to handle in your real loaders (this is where benchmarks silently go wrong):

* **Daily Tmax from 6-hourly output underestimates the true daily max** (AIFS and Aurora are 6-hourly). Either accept it consistently for the truth too (e.g. ERA5 6-hourly max), or verify against a truth built the same way. Same issue for any model with a different output frequency.
* **Precipitation**: de-accumulate if cumulative, convert m to mm, check which variable each model really predicts (not all of them do).
* **Regrid conservatively** to one grid (precip) and bilinearly (temperature). Match land masks.
* **Same initial dates and initial times** for all models; document the initial-condition source.
* Use `float32` and `dask`/`zarr` chunked on `init_time` for real ensembles (50 members x 15 leads x years is big).

A real loader looks like:

```python
@register(FORECASTS, "AIFS-ENS")
def load_aifs(inits, max_lead):
    ds = xr.open_zarr("path/to/aifs_ens.zarr")      # your storage
    ds = ds.sel(init_time=inits).isel(lead_day=slice(0, max_lead))
    # ... daily aggregation, unit conversion, rename, regrid ...
    return ds[["tmax", "tp"]]
```

Below, the loaders are **synthetic stand-ins** with a known truth, so the benchmark can be unit-checked.

In [ ]:
FORECASTS, OBS = {}, {}

def register(registry, name):
    def deco(fn):
        registry[name] = fn
        return fn
    return deco

DAY = np.timedelta64(1, "D")
LAT = np.arange(CFG["lat"][0], CFG["lat"][1] + 1e-6, 1.0)
LON = np.arange(CFG["lon"][0], CFG["lon"][1] + 1e-6, 1.0)

def _smooth_noise(shape, sigma, r):
    """Spatially-correlated unit-variance noise; the last two axes are lat/lon."""
    x = r.standard_normal(shape).astype("float32")
    x = gaussian_filter(x, (0,) * (len(shape) - 2) + (sigma, sigma), mode="nearest")
    return x / x.std()

# ---- transformations from a latent standard-normal field z to physical variables ----
def _season(dates):                      # dates: DataArray of datetimes
    doy = dates.dt.dayofyear
    yr = dates.dt.year
    return 18 + 10 * np.cos(2 * np.pi * (doy - 200) / 365.25) + 0.03 * (yr - 2005)

def z_to_tmax(z, dates):
    lat_term = -0.4 * (z.latitude - 46.0)
    return (_season(dates) + lat_term + 3.5 * z).astype("float32")

def z_to_tp(z):
    return np.maximum(0, np.exp(0.9 * z + 0.9) - 1.5).astype("float32")

_SYN = {}

@register(OBS, "synthetic")
def load_obs_synthetic():
    r = np.random.default_rng(1)
    t = pd.date_range("1991-01-01", "2025-12-31")
    shape = (len(t), len(LAT), len(LON))
    zs = {}
    for key, phi in (("t", 0.75), ("p", 0.5)):          # AR(1) in time, smooth in space
        z = lfilter([np.sqrt(1 - phi**2)], [1, -phi], _smooth_noise(shape, 1.0, r), axis=0)
        zs[key] = xr.DataArray((z / z.std()).astype("float32"), dims=("time", "latitude", "longitude"),
                               coords=dict(time=t, latitude=LAT, longitude=LON))
    _SYN.update(zs)
    return xr.Dataset(dict(tmax=z_to_tmax(zs["t"], zs["t"].time), tp=z_to_tp(zs["p"])))

SYN_MODELS = {   # purely illustrative skill/spread/tail-damping differences
    "IFS-ENS":    dict(n_members=24, tau=7.0, spread=1.00, tail=1.00),
    "AIFS-ENS":   dict(n_members=30, tau=7.5, spread=0.90, tail=0.75),
    "Aurora-ENS": dict(n_members=20, tau=6.0, spread=0.80, tail=0.65),
    "altas":      dict(n_members=16, tau=5.0, spread=0.95, tail=0.85),
}

def make_synthetic_forecast(inits, max_lead, n_members, tau, spread, tail, seed):
    r = np.random.default_rng(seed)
    I, L, M = len(inits), max_lead, n_members
    leads = np.arange(L)
    coords = dict(init_time=inits, lead_day=leads)
    valid = xr.DataArray(inits.values[:, None] + leads[None, :] * DAY, dims=("init_time", "lead_day"), coords=coords)
    rho = np.exp(-(leads + 1) / tau)[None, :, None, None, None]
    out = {}
    for key, fn in (("t", "tmax"), ("p", "tp")):
        z = _SYN[key].sel(time=valid).values[:, :, None]                      # (I, L, 1, Y, X)
        eps = _smooth_noise((I, L, M, len(LAT), len(LON)), 1.0, r)
        zm = rho * z + np.sqrt(1 - rho**2) * spread * eps
        zm = np.where(zm > 1, 1 + tail * (zm - 1), zm)                         # AI-style damping of the upper tail
        zm = xr.DataArray(zm, dims=("init_time", "lead_day", "member", "latitude", "longitude"),
                          coords=dict(**coords, member=np.arange(M), latitude=LAT, longitude=LON))
        out[fn] = z_to_tmax(zm, valid) if fn == "tmax" else z_to_tp(zm)
    return xr.Dataset(out)

for i, (name, kw) in enumerate(SYN_MODELS.items()):
    FORECASTS[name] = (lambda inits, max_lead, kw=kw, i=i: make_synthetic_forecast(inits, max_lead, seed=100 + i, **kw))

### Load everything

`INITS` are the forecast start dates. Neighbouring daily initialisations are strongly correlated, so the effective sample size is smaller than the number of forecasts. That is handled by the block bootstrap later. For real data, daily or every-2-days inits over several summers is a reasonable start.

In [ ]:
INITS = pd.DatetimeIndex(list(pd.date_range("2022-06-01", "2022-08-31")) + list(pd.date_range("2023-06-01", "2023-08-31")),
                         name="init_time")
OBS_NAME = "synthetic"
MODELS = list(SYN_MODELS)                      # e.g. ["IFS-ENS", "AIFS-ENS", "Aurora-ENS", "altas"]

obs = OBS[OBS_NAME]()
fcs = {m: FORECASTS[m](INITS, CFG["max_lead"]) for m in MODELS}
print({m: dict(f.sizes) for m, f in fcs.items()})

### Equalise ensemble size

Brier and CRPS of an ensemble depend on its size. Two safeguards, used together:

1. **Subsample** all models to the same N (random members, fixed seed).
2. Use the **fair** versions of Brier and CRPS (section 4), which are unbiased estimators of the score of an infinite ensemble.

Always report which N you used. Any N-dependent metric (hit rate at a probability threshold, reliability) needs the common N. Skip this step only if you want the "as operated" comparison (IFS 51 vs AIFS 50 vs ...), and say so.

In [ ]:
def common_members(fcs, n=None, seed=0):
    n = n or min(f.sizes["member"] for f in fcs.values())
    r = np.random.default_rng(seed)
    out = {}
    for m, f in fcs.items():
        idx = np.sort(r.choice(f.sizes["member"], n, replace=False))
        out[m] = f.isel(member=idx).assign_coords(member=np.arange(n))
    return out, n

fcs, N_MEM = common_members(fcs, CFG["n_members"])
print("common ensemble size:", N_MEM)

## 2. Local seasonal thresholds

"Extreme" = exceeding the local, day-of-year-specific percentile of the reference climatology (here ±7 days around each calendar day, so ~15 days x 30 years = 450 samples per grid point).

* For **precipitation** the percentile is computed on **wet days only** (>= 1 mm), so "top 5 % rain" means "top 5 % of rainy days".
* The threshold comes from the **observations**. If the model has a bias, it will show up as a loss of skill. This is deliberate for an "operational usefulness" benchmark. For a pure "does the model rank situations correctly" question you would instead use **model-specific thresholds** (needs reforecasts or a long model climatology) or quantile-map the forecasts first. Decide this up-front and report it.
* Leap days are folded into 28 Feb for the day-of-year axis.

In [ ]:
def doy365(t):
    """Day of year on a 365-day calendar (29 Feb merged into 28 Feb). t: DataArray of datetimes."""
    d = t.dt.dayofyear
    return (d - (t.dt.is_leap_year & (d >= 60))).astype(int)

def doy_threshold(da, q, ref, window, wet_thr=None, min_n=30):
    ref_da = da.sel(time=slice(*ref))
    doy = doy365(ref_da.time).values
    arr = ref_da.values.astype("float32")
    if wet_thr is not None:
        arr = np.where(arr >= wet_thr, arr, np.nan)
    out = np.full((365,) + arr.shape[1:], np.nan, "float32")
    for d in range(1, 366):
        dist = np.abs(doy - d)
        sel = np.minimum(dist, 365 - dist) <= window
        x = arr[sel]
        thr = np.nanquantile(x, q, axis=0)
        thr[np.isfinite(x).sum(0) < min_n] = np.nan      # too few samples (e.g. dry regions)
        out[d - 1] = thr
    return xr.DataArray(out, dims=("dayofyear", "latitude", "longitude"),
                        coords=dict(dayofyear=np.arange(1, 366), latitude=da.latitude, longitude=da.longitude))

def thr_at(thr, dates):
    """Look the threshold up for arbitrary (multi-dimensional) valid dates."""
    return thr.sel(dayofyear=doy365(dates))

## 3. Event definitions

An `EventDef` turns a physical field + its threshold into a binary indicator. Two things matter:

* the indicator is computed **per ensemble member** (so the forecast probability is the fraction of members), and on the observations with the *same* function, so the definitions can't drift apart;
* the heatwave is a **run** of >= 3 consecutive exceedance days. A day counts as "heatwave day" if it belongs to such a run. Runs are detected **inside each forecast window** (along the lead axis) for the forecast *and* for the observations, so both are truncated identically (leads 0-1 can't see a run that started before the init date). The climatological probability used as BSS reference is computed on the full observed series, so BSS at leads 0-1 is slightly pessimistic.

The two chosen events are not the only choice. `tmax_p99` below shows how to add a rarer tier in one line.

In [ ]:
def in_run(exc, dim, n):
    """1 if the day belongs to >= n consecutive exceedance days (windows may start up to n-1 days earlier)."""
    w = exc.rolling({dim: n}, min_periods=n).min()           # 1 if the n-day window ending here is all exceedances
    return xr.concat([w.shift({dim: -k}) for k in range(n)], "k").max("k", skipna=True)

@dataclass
class EventDef:
    name: str
    var: str
    q: float
    unit: str
    wet_thr: Optional[float] = None
    run_days: Optional[int] = None
    tail_kind: str = "diff"          # how to normalise the tail in QQ plots: "diff" (y - thr) or "ratio" (y / thr)

    def threshold(self, obs):
        return doy_threshold(obs[self.var], self.q, CFG["ref_period"], CFG["window"], self.wet_thr)

    def indicator(self, x, thr, dim):
        exc = x > thr
        if self.wet_thr is not None:
            exc = exc & (x >= self.wet_thr)
        exc = exc.astype("float32").where(x.notnull() & thr.notnull())
        return in_run(exc, dim, self.run_days) if self.run_days else exc

EVENTS = {
    "heatwave": EventDef("heatwave", "tmax", CFG["q"], "°C", run_days=CFG["heat_days"]),
    "rain_p95": EventDef("rain_p95", "tp", CFG["q"], "mm/day", wet_thr=CFG["wet_thr"], tail_kind="ratio"),
    # "tmax_p99": EventDef("tmax_p99", "tmax", 0.99, "°C"),         # <- a rarer single-day tier
}

### Build the verification sample

For each event and model we assemble: member indicators `pm`, observed indicator `o`, raw continuous values `y`, `y_obs`, the threshold at the valid date, and the climatological event probability `p_clim` (used as the reference forecast in the Brier skill score). Samples whose **valid date** is outside the season are masked with NaN.

In [ ]:
def build_verification(ev, obs, fcs):
    thr = ev.threshold(obs)
    ind_obs = ev.indicator(obs[ev.var], thr_at(thr, obs.time), "time")
    ref = ind_obs.sel(time=slice(*CFG["ref_period"]))
    p_clim = ref.where(ref.time.dt.month.isin(CFG["season_months"])).mean("time")      # (lat, lon)

    data = {}
    for m, fc in fcs.items():
        valid = (fc.init_time + fc.lead_day * DAY).rename("valid_time")                 # (init_time, lead_day)
        season = valid.dt.month.isin(CFG["season_months"])
        thr_f = thr_at(thr, valid)
        x = fc[ev.var]
        drop = dict(errors="ignore")
        y_obs = obs[ev.var].sel(time=valid).drop_vars("time", **drop)
        data[m] = dict(
            pm=ev.indicator(x, thr_f, "lead_day").where(season),
            # same function, same lead-axis window => identical run-truncation for forecast and truth
            o=ev.indicator(y_obs, thr_f, "lead_day").where(season),
            y=x.where(season),
            y_obs=y_obs.where(season),
            thr=thr_f.drop_vars("dayofyear", **drop).where(season),
            p_clim=p_clim,
        )
    return data, thr

VER, THR = {}, {}
for en, ev in EVENTS.items():
    VER[en], THR[en] = build_verification(ev, obs, fcs)
    d = VER[en][MODELS[0]]
    print(f"{en:9s} observed base rate in test sample: {float(d['o'].mean()):.4f}   "
          f"(climatological: {float(d['p_clim'].mean()):.4f})")

## 4. Metrics

Design rule: **every metric returns one number per (init_time, lead_day)** after a latitude-weighted spatial mean. Averaging over inits (and bootstrapping) then happens in one place. To add a metric, write `fn(d) -> DataArray(init_time, lead_day, lat, lon)` and add it to `SAMPLE_METRICS`.

| metric | why |
|---|---|
| **fair Brier score** (`bs`) + skill vs climatology | proper score for the event probability; fair = comparable across ensemble sizes |
| **fair CRPS** (`crps`) | whole-distribution quality of the continuous variable |
| **twCRPS** (`twcrps`) | CRPS restricted to the upper tail above the event threshold, via the chaining function `v(x) = max(x, thr)`; a *proper* tail score (conditioning on the observed extreme is not) |
| **SEDI** from contingency tables | hit rate / false alarm rate summary that does not degenerate for rare events |

Contingency tables are accumulated at several **probability decision thresholds**, giving ROC-like curves and the performance diagram.

In [ ]:
SD = ("latitude", "longitude")

def sp_mean(da):
    w = np.cos(np.deg2rad(da.latitude))
    return da.weighted(w).mean(SD)

# ---------- ensemble CRPS (fair), O(M log M) via sorting ----------
def _crps_np(ens, y):
    M = ens.shape[-1]
    t1 = np.abs(ens - y[..., None]).mean(-1)
    xs = np.sort(ens, axis=-1)
    k = np.arange(1, M + 1)
    t2 = ((2 * k - M - 1) * xs).sum(-1) / (M * (M - 1))         # = 1/(2M(M-1)) * sum_ij |xi - xj|
    return t1 - t2

def crps_fair(ens, y):
    return xr.apply_ufunc(_crps_np, ens, y, input_core_dims=[["member"], []])

# quick unit check against the brute-force double sum
_e = np.random.default_rng(0).normal(size=(5, 12)); _y = np.random.default_rng(1).normal(size=5)
_bf = np.abs(_e - _y[:, None]).mean(1) - np.abs(_e[:, :, None] - _e[:, None, :]).sum((1, 2)) / (2 * 12 * 11)
assert np.allclose(_crps_np(_e, _y), _bf)

# ---------- per-sample metrics ----------
def m_bs(d):                       # fair Brier score of the member fraction
    M = d["pm"].sizes["member"]
    p = d["pm"].mean("member")
    return (p - d["o"]) ** 2 - p * (1 - p) / (M - 1)

def m_bs_ref(d):                   # Brier score of the climatological probability
    return (d["p_clim"] - d["o"]) ** 2

def m_crps(d):
    return crps_fair(d["y"], d["y_obs"])

def m_twcrps(d):                   # upper-tail twCRPS, threshold = event threshold
    return crps_fair(np.maximum(d["y"], d["thr"]), np.maximum(d["y_obs"], d["thr"]))

SAMPLE_METRICS = dict(bs=m_bs, bs_ref=m_bs_ref, crps=m_crps, twcrps=m_twcrps)   # <- add yours here

# ---------- contingency tables ----------
def contingency(d, pths):
    p = d["pm"].mean("member"); o = d["o"]
    ok = p.notnull() & o.notnull()
    ev = o == 1
    tabs = []
    for t in pths:
        yes = p >= t - 1e-6
        cells = [(yes & ev & ok), (~yes & ev & ok), (yes & ~ev & ok), (~yes & ~ev & ok)]
        tabs.append(xr.concat([c.sum(SD) for c in cells], dim=pd.Index(["hits", "misses", "fa", "cn"], name="cell")))
    return xr.concat(tabs, dim=pd.Index(pths, name="pth"))

def score_model(d, pths):
    ds = xr.Dataset({k: sp_mean(fn(d)) for k, fn in SAMPLE_METRICS.items()})
    ds["counts"] = contingency(d, pths)
    return ds.compute()

def sedi(h, f, eps=1e-6):
    h, f = np.clip(h, eps, 1 - eps), np.clip(f, eps, 1 - eps)
    return (np.log(f) - np.log(h) - np.log1p(-f) + np.log1p(-h)) / (np.log(f) + np.log(h) + np.log1p(-f) + np.log1p(-h))

SCORES = {en: {m: score_model(VER[en][m], CFG["pths"]) for m in MODELS} for en in EVENTS}
print("scored:", {en: list(v) for en, v in SCORES.items()})

## 5. Uncertainty: block bootstrap over init dates, paired across models

Extreme events are rare and consecutive inits are autocorrelated, so a plain bootstrap is far too optimistic. We resample **blocks of consecutive init dates** (block >= the autocorrelation scale, ~1 week), with the **same resamples for every model**. Differences between models are then *paired*: the confidence interval of `BSS(model) - BSS(baseline)` is much tighter than comparing two independent intervals, and it is the right test for "is model A better than B?". Row 0 of every bootstrap array is the point estimate.

In [ ]:
def block_idx(n, block, B, r):
    nb = int(np.ceil(n / block))
    starts = r.integers(0, n - block + 1, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(block)).reshape(B, -1)[:, :n]
    return np.vstack([np.arange(n), idx])                        # row 0 = no resampling

def aggregate(ds, idx):
    out = {k: np.stack([np.nanmean(ds[k].values[i], axis=0) for i in idx]) for k in SAMPLE_METRICS}   # (B+1, L)
    out["bss"] = 1 - out["bs"] / out["bs_ref"]
    c = ds["counts"].transpose("init_time", "lead_day", "pth", "cell").values.astype("float64")
    cs = np.stack([c[i].sum(0) for i in idx])                                                          # (B+1, L, P, 4)
    h, mi, fa, cn = (cs[..., k] for k in range(4))
    out["pod"] = h / (h + mi)
    out["far"] = fa / (fa + cn)
    out["sr"] = h / (h + fa)
    out["sedi"] = np.where(h + fa == 0, np.nan, sedi(out["pod"], out["far"]))                          # NaN if never forecast
    return out

n_init = SCORES["heatwave"][MODELS[0]].sizes["init_time"]
IDX = block_idx(n_init, CFG["block"], CFG["n_boot"], np.random.default_rng(7))
RES = {en: {m: aggregate(SCORES[en][m], IDX) for m in MODELS} for en in EVENTS}
LEADS = SCORES["heatwave"][MODELS[0]].lead_day.values
PTH_I = CFG["pths"].index(CFG["pth_main"])

## 6. Results

### 6.1 Skill vs lead time (with the "skill horizon")

BSS = 1 - BS/BS_climatology. The lead at which it drops to 0 is the extreme-event **skill horizon**. Bands are 95 % block-bootstrap intervals.

In [ ]:
def lead_plot(ax, res, key, pth_i=None, ref=None, ylabel=""):
    ref_a = None
    if ref:
        ref_a = res[ref][key] if pth_i is None else res[ref][key][..., pth_i]
    for m, r in res.items():
        if m == ref:
            continue
        a = r[key] if pth_i is None else r[key][..., pth_i]
        if ref_a is not None:
            a = a - ref_a                                     # paired difference: same bootstrap resamples
        lo, hi = np.nanpercentile(a[1:], [2.5, 97.5], axis=0)
        ax.plot(LEADS, a[0], marker="o", ms=3, label=m)
        ax.fill_between(LEADS, lo, hi, alpha=0.15)
    ax.set_xlabel("lead (days)"); ax.set_ylabel(ylabel)
    if ref: ax.axhline(0, color="k", lw=0.8)

fig, axs = plt.subplots(len(EVENTS), 3, figsize=(15, 4 * len(EVENTS)), squeeze=False)
for row, en in enumerate(EVENTS):
    lead_plot(axs[row, 0], RES[en], "bss", ylabel="BSS (vs climatology)"); axs[row, 0].axhline(0, color="k", lw=.8)
    axs[row, 0].set_title(f"{en}: Brier skill score")
    lead_plot(axs[row, 1], RES[en], "sedi", pth_i=PTH_I, ylabel="SEDI"); axs[row, 1].set_title(f"{en}: SEDI (p >= {CFG['pth_main']})")
    lead_plot(axs[row, 2], RES[en], "twcrps", ylabel="twCRPS (lower is better)"); axs[row, 2].set_title(f"{en}: upper-tail CRPS")
axs[0, 0].legend(); plt.tight_layout(); plt.show()

### 6.2 Paired comparison against the baseline

Difference in BSS with respect to the baseline model. Where the band excludes 0, the difference is significant at the chosen block-bootstrap level.

In [ ]:
fig, axs = plt.subplots(1, len(EVENTS), figsize=(6.5 * len(EVENTS), 4), squeeze=False)
for ax, en in zip(axs[0], EVENTS):
    lead_plot(ax, RES[en], "bss", ref=CFG["baseline"], ylabel=f"ΔBSS vs {CFG['baseline']}")
    ax.set_title(en)
axs[0, 0].legend(); plt.tight_layout(); plt.show()

def summary_table(en, leads, pth_i=PTH_I):
    sl = np.isin(LEADS, leads)
    rows = {}
    for m in MODELS:
        r = RES[en][m]
        f = lambda a: np.nanmean(a[:, sl], axis=1)
        bss, sd, tw = f(r["bss"]), f(r["sedi"][..., pth_i]), f(r["twcrps"])
        ci = lambda a: f"{a[0]:.3f} [{np.nanpercentile(a[1:], 2.5):.3f}, {np.nanpercentile(a[1:], 97.5):.3f}]"
        rows[m] = dict(BSS=ci(bss), SEDI=ci(sd), twCRPS=ci(tw))
    return pd.DataFrame(rows).T

for en in EVENTS:
    print(f"\n{en}: leads 2-4 days, N={N_MEM} members"); display(summary_table(en, [2, 3, 4]))

### 6.3 Reliability and performance diagram

* **Reliability**: when the ensemble says p, does the event happen with frequency p? Rare events live in the low-probability bins, so the sharpness histogram (log scale) is as informative as the curve. Only bins with >= 20 samples are drawn.
* **Performance diagram**: POD vs success ratio along the probability-threshold sweep. Closer to the top right is better; dashed lines are frequency bias, grey curves are CSI.

In [ ]:
def reliability(d, leads, nb=10):
    p = d["pm"].mean("member").sel(lead_day=leads).values.ravel()
    o = d["o"].sel(lead_day=leads).values.ravel()
    ok = np.isfinite(p) & np.isfinite(o); p, o = p[ok], o[ok]
    b = np.clip(np.digitize(p, np.linspace(0, 1, nb + 1)) - 1, 0, nb - 1)
    cnt = np.bincount(b, minlength=nb)
    of = np.bincount(b, weights=o, minlength=nb) / np.maximum(cnt, 1)
    pf = np.bincount(b, weights=p, minlength=nb) / np.maximum(cnt, 1)
    return pf, of, cnt

def performance_diagram(ax, SC, leads):
    g = np.linspace(0.01, 1, 100); X, Y = np.meshgrid(g, g)
    ax.contour(X, Y, 1 / (1 / X + 1 / Y - 1), levels=np.arange(.1, 1, .1), colors="gray", linewidths=.5)
    for b in (0.5, 1, 2):
        ax.plot(g, np.clip(b * g, 0, 1), "k--", lw=.5)
    for m, ds in SC.items():
        c = ds.counts.sel(lead_day=leads).sum(("init_time", "lead_day"))
        h, mi, fa = c.sel(cell="hits"), c.sel(cell="misses"), c.sel(cell="fa")
        ax.plot(h / (h + fa), h / (h + mi), "-o", ms=3, label=m)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel("success ratio (1 - FAR)"); ax.set_ylabel("POD")

LEAD_WIN = [2, 3, 4]
fig, axs = plt.subplots(len(EVENTS), 3, figsize=(16, 4.5 * len(EVENTS)), squeeze=False)
for row, en in enumerate(EVENTS):
    ax = axs[row, 0]
    ax.plot([0, 1], [0, 1], "k--", lw=.8)
    for m in MODELS:
        pf, of, cnt = reliability(VER[en][m], LEAD_WIN); k = cnt >= 20
        ax.plot(pf[k], of[k], "-o", ms=3, label=m)
    ax.set_xlabel("forecast probability"); ax.set_ylabel("observed frequency"); ax.set_title(f"{en}: reliability (leads {LEAD_WIN})")
    ax = axs[row, 1]
    w = 0.8 / len(MODELS)
    for j, m in enumerate(MODELS):
        _, _, cnt = reliability(VER[en][m], LEAD_WIN)
        ax.bar(np.arange(10) + j * w, cnt, w, label=m)
    ax.set_yscale("log"); ax.set_title("sharpness (counts)"); ax.set_xlabel("probability bin")
    performance_diagram(axs[row, 2], SCORES[en], LEAD_WIN); axs[row, 2].set_title(f"{en}: performance diagram")
axs[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

### 6.4 Is the tail damped? Upper-tail QQ

Forecast vs observed quantiles of the **excess over the threshold** (`y - thr` for temperature, `y / thr` for rain), pooled over space, init dates and members. Points under the 1:1 line mean the model **underestimates its extremes** (the classic smoothing problem of ML models). A calibrated ensemble sits on the 1:1 line; a curve that bends below it in the tail is the signature of damped extremes (in this synthetic example, the injected tail damping is recovered, strongest for the fake `Aurora-ENS`).

In [ ]:
def tail_qq(d, ev, leads, qs):
    norm = (lambda y, t: y - t) if ev.tail_kind == "diff" else (lambda y, t: y / t)
    yo = norm(d["y_obs"], d["thr"]).sel(lead_day=leads).values.ravel()
    yf = norm(d["y"], d["thr"]).sel(lead_day=leads).values.ravel()
    return np.nanquantile(yo, qs), np.nanquantile(yf, qs)

QS = np.array([0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999])
fig, axs = plt.subplots(1, len(EVENTS), figsize=(6 * len(EVENTS), 4.5), squeeze=False)
for ax, (en, ev) in zip(axs[0], EVENTS.items()):
    lo, hi = np.inf, -np.inf
    for m in MODELS:
        qo, qf = tail_qq(VER[en][m], ev, [1, 2, 3], QS)
        ax.plot(qo, qf, "-o", ms=3, label=m); lo, hi = min(lo, qo.min(), qf.min()), max(hi, qo.max(), qf.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=.8)
    ax.set_xlabel("observed quantile"); ax.set_ylabel("forecast quantile (all members)")
    ax.set_title(f"{en}: upper-tail QQ ({'y - thr' if ev.tail_kind == 'diff' else 'y / thr'}, leads 1-3)")
axs[0, 0].legend(); plt.tight_layout(); plt.show()

### 6.5 Rain: neighbourhood verification (FSS)

Point-wise scores punish a displaced rain band twice. The **Fractions Skill Score** asks at which spatial scale the forecast becomes useful: the smallest scale where FSS exceeds `0.5 + f/2` (f = observed event frequency) is the "skillful scale". Here the neighbourhood is in grid cells (multiply by your grid spacing for km); fractions are computed per member and the squared errors are pooled over inits and members, which keeps it stable for rare events.

In [ ]:
def fss_curve(d, scales, leads):
    res = np.full((len(leads), len(scales)), np.nan)
    for i, l in enumerate(leads):
        F = d["pm"].sel(lead_day=l).transpose("init_time", "member", "latitude", "longitude").values
        O = d["o"].sel(lead_day=l).transpose("init_time", "latitude", "longitude").values[:, None]
        v = (np.isfinite(F) & np.isfinite(O)).astype("float32")
        F0, O0 = np.nan_to_num(F) * v, np.nan_to_num(O) * v
        for j, n in enumerate(scales):
            flt = lambda a: uniform_filter(a, size=(1, 1, n, n), mode="constant")
            cov = flt(v); ok = cov > 0
            pf, po = flt(F0)[ok] / cov[ok], flt(O0)[ok] / cov[ok]
            res[i, j] = 1 - ((pf - po) ** 2).sum() / ((pf**2).sum() + (po**2).sum())
    return res

SCALES = [1, 3, 5, 7, 9, 13]
FSS_LEADS = [1, 3, 5]
fig, axs = plt.subplots(1, len(FSS_LEADS), figsize=(5 * len(FSS_LEADS), 4), sharey=True)
f_obs = float(VER["rain_p95"][MODELS[0]]["o"].mean())
for ax, (i, l) in zip(axs, enumerate(FSS_LEADS)):
    for m in MODELS:
        ax.plot(SCALES, fss_curve(VER["rain_p95"][m], SCALES, [l])[0], "-o", ms=3, label=m)
    ax.axhline(0.5 + f_obs / 2, color="k", ls="--", lw=.8, label="useful: 0.5 + f/2")
    ax.set_xlabel("neighbourhood size (grid cells)"); ax.set_title(f"rain_p95, lead {l} d")
axs[0].set_ylabel("FSS"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### 6.6 Where is the skill? Maps of BSS

Spatial structure of skill at one lead window. In real data look for coastal/orographic patterns (Mediterranean heavy rain, Alps/Pyrenees) and compare land/sea.

In [ ]:
def bss_map(d, leads):
    sl = dict(lead_day=leads)
    bs = m_bs(d).sel(**sl).mean(("init_time", "lead_day"))
    ref = m_bs_ref(d).sel(**sl).mean(("init_time", "lead_day"))
    return 1 - bs / ref

for en in EVENTS:
    fig, axs = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.4), sharey=True)
    for ax, m in zip(axs, MODELS):
        mp = bss_map(VER[en][m], [2, 3, 4])
        im = ax.pcolormesh(mp.longitude, mp.latitude, mp.values, vmin=-0.5, vmax=0.5, cmap="RdBu", shading="auto")
        ax.set_title(f"{m}"); ax.set_xlabel("lon")
    axs[0].set_ylabel("lat"); fig.suptitle(f"{en}: BSS, leads 2-4 d", y=1.02)
    fig.colorbar(im, ax=axs, shrink=0.8, label="BSS"); plt.show()

### 6.7 Case study: ensemble plume at a point

Aggregate statistics tell you *how often*; case studies tell you *why*. Pick a known event (e.g. the August 2023 French heatwave, the Oct 2024 Mediterranean floods), choose an init some days ahead, and plot members, the observation and the threshold. With real data, add Z500 / IVT maps to check the event is predicted **for the right reason**.

In [ ]:
def plume(ax, ev, model, init, lat, lon):
    f = fcs[model][ev.var].sel(init_time=init).sel(latitude=lat, longitude=lon, method="nearest")
    valid = pd.Timestamp(init) + pd.to_timedelta(f.lead_day.values, "D")
    vt = xr.DataArray(valid, dims="v")
    thr = thr_at(THR[ev.name], vt).sel(latitude=lat, longitude=lon, method="nearest")
    o = obs[ev.var].sel(time=valid).sel(latitude=lat, longitude=lon, method="nearest")
    ax.plot(valid, f.transpose("lead_day", "member").values, color="tab:blue", alpha=0.25, lw=0.8)
    ax.plot(valid, o.values, "k-o", ms=3, label="obs"); ax.plot(valid, thr.values, "r--", label=f"p{int(ev.q*100)} threshold")
    ax.set_title(f"{model}, init {pd.Timestamp(init):%Y-%m-%d}"); ax.set_ylabel(f"{ev.var} [{ev.unit}]"); ax.tick_params(axis="x", rotation=30)

# pick the init with the largest observed heatwave coverage in the test sample, as a stand-in for a famous case
cover = VER["heatwave"][MODELS[0]]["o"].isel(lead_day=4).mean(SD)
case_init = cover.init_time.values[int(np.nanargmax(cover.values))]
print("case init:", pd.Timestamp(case_init).date())
PARIS = (48.85, 2.35)
fig, axs = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 3.6), sharey=True)
for ax, m in zip(axs, MODELS):
    plume(ax, EVENTS["heatwave"], m, case_init, *PARIS)
axs[0].legend(); plt.tight_layout(); plt.show()

## 7. How to adapt this

**Add a model.** Write a loader that returns the data contract of section 1 and add its name to `MODELS`. Nothing else changes. To compare against the "as operated" ensemble size, set `n_members` to a number larger than the smallest ensemble, or skip `common_members` (then rely on the fair scores only).

**Change the truth.** Write an `OBS[...]` loader. Good choices for France: ERA5 (quick start, but it smooths peaks and its precipitation is a model product), E-OBS or station data for temperature, COMEPHORE / ANTILOPE radar-gauge or IMERG/MSWEP for rain. Recompute the thresholds from the *same* product used as truth. Evaluating with ERA5 as truth favours models trained on or initialised from ERA5.

**Change the event.** Add an `EventDef` (e.g. a 99th-percentile tier, a 72 h rain accumulation by pre-aggregating the daily field, or a Tmin-based tropical-night count). For real heatwave *objects* (location / extent / duration / onset error) you need an object-matching step on the indicator fields. Not included here, but the `VER[...]['o']` and `['pm']` arrays are the right inputs.

**Add a metric.** Per-sample metric: add `fn(d)` to `SAMPLE_METRICS`; bootstrap and plotting plumbing for `lead_plot` work as-is. Contingency-based metric: compute it from `ds.counts` in `aggregate`.

**Checklist before believing a result**

* Test period is after every model's training period (or all models share one common test window).
* Same inits, same grid, same truth, same ensemble size.
* Threshold source (observations vs model climatology) is stated.
* Verification of heatwaves is not conditioned on the observed outcome.
* Confidence intervals come from a block bootstrap, and model differences are paired.
* Raw vs post-processed (EMOS, quantile mapping) is stated. AI models often gain more from calibration than the IFS does.

**Natural extensions**: station-based verification, relative economic value curves, return-period events with extreme-value fits, physical-consistency diagnostics (Z500 blocking, soil moisture for heatwaves, IVT for rain), and smoothing / power-spectrum diagnostics with lead time.